In [1]:
# Cell 1: installs if needed
# Uncomment if needed
# !pip install scanpy anndata pandas scipy numpy

import os
import re
import tarfile
import zipfile
import gzip
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse, io

In [2]:
# Cell 2: paths
BASE = Path()   # change if needed
RAW_TAR = BASE / "GSE185477_RAW.tar"
META_GZ = BASE / "GSE185477_Final_Metadata.txt.gz"
EXTRA_SC_ZIP = BASE / "GSE185477_GSM3178784_C41_SC_raw_counts.zip"

EXTRACT_DIR = BASE / "extracted"
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE.resolve())
print("RAW_TAR exists:", RAW_TAR.exists())
print("META_GZ exists:", META_GZ.exists())
print("EXTRA_SC_ZIP exists:", EXTRA_SC_ZIP.exists())

BASE: C:\Users\ankit\Documents\scFM\train_data\GSE185477
RAW_TAR exists: True
META_GZ exists: True
EXTRA_SC_ZIP exists: True


In [4]:
# Cell 3: extract the tar and any nested zips

def safe_extract_tar(tar_path: Path, out_dir: Path):
    with tarfile.open(tar_path, "r") as tar:
        tar.extractall(out_dir)

def unzip_all(root_dir: Path, rounds: int = 3):
    """
    Recursively unzip all .zip files found under root_dir.
    Multiple rounds helps in case zip files contain more zip files.
    """
    for _ in range(rounds):
        zip_files = list(root_dir.rglob("*.zip"))
        if not zip_files:
            break
        for zf in zip_files:
            out = zf.with_suffix("")  # remove .zip
            out.mkdir(parents=True, exist_ok=True)
            try:
                with zipfile.ZipFile(zf, "r") as z:
                    z.extractall(out)
            except Exception as e:
                print(f"Could not unzip {zf}: {e}")

# Extract main tar
safe_extract_tar(RAW_TAR, EXTRACT_DIR)

# Extract extra donor-1 single-cell zip
extra_dir = EXTRACT_DIR / "GSM3178784_extra"
extra_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(EXTRA_SC_ZIP, "r") as z:
    z.extractall(extra_dir)

# Unzip anything nested inside the extracted files
unzip_all(EXTRACT_DIR, rounds=5)

print("Extraction complete.")

Extraction complete.


In [5]:
# Cell 4: inspect extracted files
all_files = sorted([p for p in EXTRACT_DIR.rglob("*") if p.is_file()])
print(f"Total files found: {len(all_files)}")
for p in all_files[:200]:
    print(p.relative_to(EXTRACT_DIR))

Total files found: 57
GSM3178784_extra\C41_SC\GRCh38\barcodes.tsv.gz
GSM3178784_extra\C41_SC\GRCh38\genes.tsv.gz
GSM3178784_extra\C41_SC\GRCh38\matrix.mtx.gz
GSM5615999_C41_CST_raw_counts\C41_CST\barcodes.tsv.gz
GSM5615999_C41_CST_raw_counts\C41_CST\features.tsv.gz
GSM5615999_C41_CST_raw_counts\C41_CST\matrix.mtx.gz
GSM5615999_C41_CST_raw_counts.zip
GSM5616000_C41_NST_raw_counts\C41_NST\barcodes.tsv.gz
GSM5616000_C41_NST_raw_counts\C41_NST\features.tsv.gz
GSM5616000_C41_NST_raw_counts\C41_NST\matrix.mtx.gz
GSM5616000_C41_NST_raw_counts.zip
GSM5616001_C41_TST_raw_counts\C41_TST\barcodes.tsv.gz
GSM5616001_C41_TST_raw_counts\C41_TST\features.tsv.gz
GSM5616001_C41_TST_raw_counts\C41_TST\matrix.mtx.gz
GSM5616001_C41_TST_raw_counts.zip
GSM5616002_C58_SC_raw_counts\C58_SC\barcodes.tsv.gz
GSM5616002_C58_SC_raw_counts\C58_SC\C58_3pr_gt250_barcodes.txt
GSM5616002_C58_SC_raw_counts\C58_SC\features.tsv.gz
GSM5616002_C58_SC_raw_counts\C58_SC\matrix.mtx.gz
GSM5616002_C58_SC_raw_counts.zip
GSM5616003

In [6]:
# Cell 5: read metadata file if you want to inspect it
# This is optional but useful for checking available columns.

meta = pd.read_csv(META_GZ, sep="\t", compression="gzip", low_memory=False)
print(meta.shape)
print(meta.columns.tolist()[:50])
display(meta.head())

(73295, 19)
['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'cell_barcode', 'donor', 'cell_ID', 'S.Score', 'G2M.Score', 'Phase', 'sample', 'assay_type', 'Coarse_clusters', 'Manual_Annotation', 'Subcluster_Group', 'sub_cluster', 'sub_annotation', 'UMAP1', 'UMAP2']


,orig.ident,nCount_RNA,nFeature_RNA,percent.mt,cell_barcode,donor,cell_ID,S.Score,G2M.Score,Phase,sample,assay_type,Coarse_clusters,Manual_Annotation,Subcluster_Group,sub_cluster,sub_annotation,UMAP1,UMAP2
C41_AAACCTGAGCCTTGAT,C41,428,199,32.087912,AAACCTGAGCCTTGAT,C41,C41_AAACCTGAGCCTTGAT,-0.009246,-0.016106,G1,C41,single_cell,11,InterHep,Hepatocyte,Hep_6,Hep_Unidentified,-3.321926,-6.397605
C41_AAACCTGAGGTCATCT,C41,275,181,19.666667,AAACCTGAGGTCATCT,C41,C41_AAACCTGAGGTCATCT,-0.018254,-0.027253,G1,C41,single_cell,10,cvLSECs,LSECs,LSEC_1,LSEC_Central Venous LSEC,14.034962,2.834506
C41_AAACCTGAGTCGCCGT,C41,632,264,15.703704,AAACCTGAGTCGCCGT,C41,C41_AAACCTGAGTCGCCGT,-0.010180,-0.027701,G1,C41,single_cell,13,Bcells,Bcells,Lymph_5,Lymph_Mature B cells,-1.928654,-7.027950
C41_AAACCTGAGTGGAGTC,C41,10508,1923,12.188988,AAACCTGAGTGGAGTC,C41,C41_AAACCTGAGTGGAGTC,0.030654,-0.030400,S,C41,single_cell,4,InterHep,Hepatocyte,Hep_4,Hep_PP2,1.582531,3.822350
C41_AAACCTGAGTTAACGA,C41,503,220,26.355140,AAACCTGAGTTAACGA,C41,C41_AAACCTGAGTTAACGA,-0.020764,-0.021055,G1,C41,single_cell,7,NKTcell,NKTcell,Lymph_0,Lymph_ab T cells,-2.318836,-13.029733


In [7]:
# Cell 6: sample annotation map
# VISIUM is intentionally excluded from the final AnnData merge.

sample_info = {
    "GSM3178784": {"sample_title": "Single cell Donor 1", "donor": "Donor_1", "assay": "scRNA-seq", "prep": "SC"},
    "GSM5615999": {"sample_title": "Single nuclei (CST) Donor 1", "donor": "Donor_1", "assay": "snRNA-seq", "prep": "CST"},
    "GSM5616000": {"sample_title": "Single nuclei (NST) Donor 1", "donor": "Donor_1", "assay": "snRNA-seq", "prep": "NST"},
    "GSM5616001": {"sample_title": "Single nuclei (TST) Donor 1", "donor": "Donor_1", "assay": "snRNA-seq", "prep": "TST"},
    "GSM5616002": {"sample_title": "Single cell Donor 2", "donor": "Donor_2", "assay": "scRNA-seq", "prep": "SC"},
    "GSM5616003": {"sample_title": "Single nuclei (TST) Donor 2", "donor": "Donor_2", "assay": "snRNA-seq", "prep": "TST"},
    "GSM5616004": {"sample_title": "Single cell Donor 3", "donor": "Donor_3", "assay": "scRNA-seq", "prep": "SC"},
    "GSM5616005": {"sample_title": "Single nuclei (TST) Donor 3", "donor": "Donor_3", "assay": "snRNA-seq", "prep": "TST"},
    "GSM5616006": {"sample_title": "Single cell Donor 4", "donor": "Donor_4", "assay": "scRNA-seq", "prep": "SC"},
    "GSM5616007": {"sample_title": "Single nuclei (TST) Donor 4", "donor": "Donor_4", "assay": "snRNA-seq", "prep": "TST"},
}

In [8]:
# Cell 7: helpers to find sample IDs and load matrices

def infer_gsm_from_path(path: Path):
    m = re.search(r"(GSM\d+)", str(path))
    if m:
        return m.group(1)
    return None

def read_table_maybe_gzip(path: Path):
    if path.suffix == ".gz":
        return pd.read_csv(path, sep=None, engine="python", compression="gzip", index_col=0)
    return pd.read_csv(path, sep=None, engine="python", index_col=0)

def load_from_10x_dir(d: Path):
    """
    Load a 10x-style directory containing matrix/barcodes/features.
    Handles gzipped or plain files.
    """
    matrix_file = None
    barcode_file = None
    feature_file = None

    for p in d.iterdir():
        name = p.name
        if re.match(r"matrix\.mtx(\.gz)?$", name):
            matrix_file = p
        elif re.match(r"barcodes\.(tsv|txt)(\.gz)?$", name):
            barcode_file = p
        elif re.match(r"(features|genes)\.(tsv|txt)(\.gz)?$", name):
            feature_file = p

    if matrix_file is None or barcode_file is None or feature_file is None:
        raise FileNotFoundError(f"Missing 10x files in {d}")

    # matrix
    if matrix_file.suffix == ".gz":
        with gzip.open(matrix_file, "rt") as f:
            X = io.mmread(f).tocsr()
    else:
        X = io.mmread(matrix_file).tocsr()

    # barcodes
    if barcode_file.suffix == ".gz":
        barcodes = pd.read_csv(barcode_file, sep="\t", header=None, compression="gzip")
    else:
        barcodes = pd.read_csv(barcode_file, sep="\t", header=None)
    barcodes = barcodes.iloc[:, 0].astype(str).tolist()

    # features/genes
    if feature_file.suffix == ".gz":
        feats = pd.read_csv(feature_file, sep="\t", header=None, compression="gzip")
    else:
        feats = pd.read_csv(feature_file, sep="\t", header=None)

    if feats.shape[1] >= 2:
        gene_ids = feats.iloc[:, 0].astype(str).tolist()
        gene_names = feats.iloc[:, 1].astype(str).tolist()
    else:
        gene_ids = feats.iloc[:, 0].astype(str).tolist()
        gene_names = feats.iloc[:, 0].astype(str).tolist()

    # 10x mtx is genes x cells, AnnData expects cells x genes
    adata = ad.AnnData(X=X.T.tocsr())
    adata.obs_names = pd.Index(barcodes).astype(str)
    adata.var_names = pd.Index(gene_names).astype(str)
    adata.var["gene_id"] = gene_ids
    adata.var["gene_symbol"] = gene_names
    adata.var_names_make_unique()
    adata.obs_names_make_unique()
    return adata

def load_from_dense_table(path: Path):
    """
    Load a dense counts table where rows=genes and columns=cells.
    """
    df = read_table_maybe_gzip(path)

    # first column already used as index; assume rows are genes
    genes = df.index.astype(str)
    cells = df.columns.astype(str)

    X = sparse.csr_matrix(df.to_numpy(dtype=np.float32).T)
    adata = ad.AnnData(X=X)
    adata.obs_names = pd.Index(cells).astype(str)
    adata.var_names = pd.Index(genes).astype(str)
    adata.var["gene_symbol"] = adata.var_names
    adata.var_names_make_unique()
    adata.obs_names_make_unique()
    return adata

def candidate_count_files(root: Path):
    """
    Find plausible count objects.
    """
    files = list(root.rglob("*"))
    out = []
    for p in files:
        if not p.is_file():
            continue
        name = p.name.lower()
        if (
            "raw_count" in name
            or "raw_counts" in name
            or re.match(r"matrix\.mtx(\.gz)?$", name)
            or name.endswith(".h5")
            or name.endswith(".csv")
            or name.endswith(".csv.gz")
            or name.endswith(".txt")
            or name.endswith(".txt.gz")
            or name.endswith(".tsv")
            or name.endswith(".tsv.gz")
        ):
            out.append(p)
    return sorted(set(out))

In [9]:
# Cell 8: discover likely sample-specific inputs

cands = candidate_count_files(EXTRACT_DIR)
for p in cands[:300]:
    print(p.relative_to(EXTRACT_DIR))

GSM3178784_extra\C41_SC\GRCh38\barcodes.tsv.gz
GSM3178784_extra\C41_SC\GRCh38\genes.tsv.gz
GSM3178784_extra\C41_SC\GRCh38\matrix.mtx.gz
GSM5615999_C41_CST_raw_counts\C41_CST\barcodes.tsv.gz
GSM5615999_C41_CST_raw_counts\C41_CST\features.tsv.gz
GSM5615999_C41_CST_raw_counts\C41_CST\matrix.mtx.gz
GSM5615999_C41_CST_raw_counts.zip
GSM5616000_C41_NST_raw_counts\C41_NST\barcodes.tsv.gz
GSM5616000_C41_NST_raw_counts\C41_NST\features.tsv.gz
GSM5616000_C41_NST_raw_counts\C41_NST\matrix.mtx.gz
GSM5616000_C41_NST_raw_counts.zip
GSM5616001_C41_TST_raw_counts\C41_TST\barcodes.tsv.gz
GSM5616001_C41_TST_raw_counts\C41_TST\features.tsv.gz
GSM5616001_C41_TST_raw_counts\C41_TST\matrix.mtx.gz
GSM5616001_C41_TST_raw_counts.zip
GSM5616002_C58_SC_raw_counts\C58_SC\barcodes.tsv.gz
GSM5616002_C58_SC_raw_counts\C58_SC\C58_3pr_gt250_barcodes.txt
GSM5616002_C58_SC_raw_counts\C58_SC\features.tsv.gz
GSM5616002_C58_SC_raw_counts\C58_SC\matrix.mtx.gz
GSM5616002_C58_SC_raw_counts.zip
GSM5616003_C58_TST_raw_counts\C5

In [10]:
# Cell 9: build a sample->path mapping
# This tries to find a suitable input for each sample.
# It prefers 10x-style matrix directories, otherwise falls back to dense count tables.

def find_sample_input(gsm_id: str, root: Path):
    sample_paths = [p for p in root.rglob("*") if gsm_id in str(p)]
    
    # Prefer directories containing matrix.mtx
    for p in sample_paths:
        if p.is_dir():
            contents = [x.name for x in p.iterdir()] if p.exists() else []
            if any(re.match(r"matrix\.mtx(\.gz)?$", x) for x in contents):
                return ("10x_dir", p)

    # Next, parent dirs of matrix files
    for p in sample_paths:
        if p.is_file() and re.match(r"matrix\.mtx(\.gz)?$", p.name):
            return ("10x_dir", p.parent)

    # Dense raw count tables
    for p in sample_paths:
        if p.is_file():
            name = p.name.lower()
            if "raw_count" in name or "raw_counts" in name:
                return ("dense", p)

    # Last resort: any csv/tsv/txt under the sample path
    for p in sample_paths:
        if p.is_file() and any(str(p).lower().endswith(ext) for ext in [".csv", ".csv.gz", ".tsv", ".tsv.gz", ".txt", ".txt.gz"]):
            return ("dense", p)

    return None, None

sample_inputs = {}
for gsm in sample_info:
    mode, path = find_sample_input(gsm, EXTRACT_DIR)
    sample_inputs[gsm] = (mode, path)

for gsm, (mode, path) in sample_inputs.items():
    print(gsm, mode, path)

GSM3178784 10x_dir extracted\GSM3178784_extra\C41_SC\GRCh38
GSM5615999 10x_dir extracted\GSM5615999_C41_CST_raw_counts\C41_CST
GSM5616000 10x_dir extracted\GSM5616000_C41_NST_raw_counts\C41_NST
GSM5616001 10x_dir extracted\GSM5616001_C41_TST_raw_counts\C41_TST
GSM5616002 10x_dir extracted\GSM5616002_C58_SC_raw_counts\C58_SC
GSM5616003 10x_dir extracted\GSM5616003_C58_TST_raw_counts\C58_TST
GSM5616004 10x_dir extracted\GSM5616004_C70_SC_raw_counts\C70_SC
GSM5616005 10x_dir extracted\GSM5616005_C70_TST_raw_counts\C70_TST
GSM5616006 10x_dir extracted\GSM5616006_C72_SC_raw_counts\C72_SC
GSM5616007 10x_dir extracted\GSM5616007_C72_TST_raw_counts\C72_TST


In [11]:
# Cell 10: load each sample into AnnData

adatas = []

for gsm, (mode, path) in sample_inputs.items():
    if path is None:
        print(f"Skipping {gsm}: no input found")
        continue

    print(f"Loading {gsm} from {path}")
    if mode == "10x_dir":
        adata = load_from_10x_dir(path)
    elif mode == "dense":
        adata = load_from_dense_table(path)
    else:
        print(f"Skipping {gsm}: unknown mode")
        continue

    # annotate
    info = sample_info[gsm]
    for k, v in info.items():
        adata.obs[k] = v
    adata.obs["gsm_id"] = gsm
    adata.obs["condition"] = "Healthy"
    adata.obs["dataset"] = "GSE185477"

    # keep raw counts in layer
    adata.layers["counts"] = adata.X.copy()

    # prepend sample ID to barcode so obs_names are globally unique
    adata.obs_names = [f"{gsm}_{bc}" for bc in adata.obs_names]
    adata.obs_names_make_unique()

    adatas.append(adata)

print(f"Loaded {len(adatas)} sample objects")

Loading GSM3178784 from extracted\GSM3178784_extra\C41_SC\GRCh38
Loading GSM5615999 from extracted\GSM5615999_C41_CST_raw_counts\C41_CST
Loading GSM5616000 from extracted\GSM5616000_C41_NST_raw_counts\C41_NST
Loading GSM5616001 from extracted\GSM5616001_C41_TST_raw_counts\C41_TST
Loading GSM5616002 from extracted\GSM5616002_C58_SC_raw_counts\C58_SC
Loading GSM5616003 from extracted\GSM5616003_C58_TST_raw_counts\C58_TST
Loading GSM5616004 from extracted\GSM5616004_C70_SC_raw_counts\C70_SC
Loading GSM5616005 from extracted\GSM5616005_C70_TST_raw_counts\C70_TST
Loading GSM5616006 from extracted\GSM5616006_C72_SC_raw_counts\C72_SC
Loading GSM5616007 from extracted\GSM5616007_C72_TST_raw_counts\C72_TST
Loaded 10 sample objects


In [12]:
# Cell 11: merge all samples

adata_merged = ad.concat(
    adatas,
    join="outer",
    merge="same",
    label="batch",
    keys=[a.obs["gsm_id"].iloc[0] for a in adatas],
    index_unique=None
)

adata_merged.var_names_make_unique()
adata_merged.obs_names_make_unique()

print(adata_merged)
print(adata_merged.obs["assay"].value_counts(dropna=False))
print(adata_merged.obs["prep"].value_counts(dropna=False))
print(adata_merged.obs["donor"].value_counts(dropna=False))

AnnData object with n_obs × n_vars = 43718400 × 45068
    obs: 'sample_title', 'donor', 'assay', 'prep', 'gsm_id', 'condition', 'dataset', 'batch'
    layers: 'counts'
assay
snRNA-seq    40769280
scRNA-seq     2949120
Name: count, dtype: int64
prep
TST    27179520
CST     6794880
NST     6794880
SC      2949120
Name: count, dtype: int64
donor
Donor_1    21121920
Donor_2     7532160
Donor_3     7532160
Donor_4     7532160
Name: count, dtype: int64


In [13]:
# Cell 12: basic QC fields

X = adata_merged.layers["counts"]

if sparse.issparse(X):
    adata_merged.obs["n_counts"] = np.asarray(X.sum(axis=1)).ravel()
    adata_merged.obs["n_genes"] = np.asarray((X > 0).sum(axis=1)).ravel()
else:
    adata_merged.obs["n_counts"] = X.sum(axis=1)
    adata_merged.obs["n_genes"] = (X > 0).sum(axis=1)

adata_merged.var["gene_symbol"] = adata_merged.var_names.astype(str)

print(adata_merged.obs[["gsm_id", "donor", "assay", "prep", "n_counts", "n_genes"]].head())

                                   gsm_id    donor      assay prep  n_counts  \
GSM3178784_AAACCTGAGAAACCAT-1  GSM3178784  Donor_1  scRNA-seq   SC         0   
GSM3178784_AAACCTGAGAAACCGC-1  GSM3178784  Donor_1  scRNA-seq   SC         0   
GSM3178784_AAACCTGAGAAACCTA-1  GSM3178784  Donor_1  scRNA-seq   SC         0   
GSM3178784_AAACCTGAGAAACGAG-1  GSM3178784  Donor_1  scRNA-seq   SC         0   
GSM3178784_AAACCTGAGAAACGCC-1  GSM3178784  Donor_1  scRNA-seq   SC         1   

                               n_genes  
GSM3178784_AAACCTGAGAAACCAT-1        0  
GSM3178784_AAACCTGAGAAACCGC-1        0  
GSM3178784_AAACCTGAGAAACCTA-1        0  
GSM3178784_AAACCTGAGAAACGAG-1        0  
GSM3178784_AAACCTGAGAAACGCC-1        1  


In [14]:
# Remove empty barcodes
adata_merged = adata_merged[(adata_merged.obs["n_counts"] > 0) & (adata_merged.obs["n_genes"] > 0)].copy()

print(adata_merged)

AnnData object with n_obs × n_vars = 9560531 × 45068
    obs: 'sample_title', 'donor', 'assay', 'prep', 'gsm_id', 'condition', 'dataset', 'batch', 'n_counts', 'n_genes'
    var: 'gene_symbol'
    layers: 'counts'


In [8]:
# Cell 13: save
out_file = BASE / "GSE185477_sc_sn_healthy_raw.h5ad"
adata_merged.write_h5ad(out_file, compression="gzip")
print(f"Saved to: {out_file.resolve()}")

Saved to: C:\Users\ankit\Documents\scFM\train_data\GSE185477\GSE185477_sc_sn_healthy_raw.h5ad


In [3]:
# Load in saved adata using full path
out_file = BASE / "GSE185477_sc_sn_healthy_raw.h5ad"
adata_merged = sc.read_h5ad(out_file)
print(adata_merged)


AnnData object with n_obs × n_vars = 9560531 × 45068
    obs: 'sample_title', 'donor', 'assay', 'prep', 'gsm_id', 'condition', 'dataset', 'batch', 'n_counts', 'n_genes', 'Sex', 'Cause_of_death'
    var: 'gene_symbol'
    layers: 'counts'


In [4]:
print(adata_merged)

AnnData object with n_obs × n_vars = 9560531 × 45068
    obs: 'sample_title', 'donor', 'assay', 'prep', 'gsm_id', 'condition', 'dataset', 'batch', 'n_counts', 'n_genes', 'Sex', 'Cause_of_death'
    var: 'gene_symbol'
    layers: 'counts'


In [5]:
# Read in the additional_meta.csv that's in the same directory and view head
additional_meta_path = BASE / "additional_meta.csv"
additional_meta = pd.read_csv(additional_meta_path)
print(additional_meta.head())

    Sample Sex   Cause_of_death
0  Donor_1   F         Aneurysm
1  Donor_2   M       Head Trama
2  Donor_3   F        Hemorrage
3  Donor_4   F  Anoxia/Drowning


In [6]:
# Merge the additional metadata into adata_merged.obs based on donor in the adata_merged.obs and the "Sample" column in additional_meta. Use a left join so we keep all cells in adata_merged.
adata_merged.obs = adata_merged.obs.merge(
    additional_meta.rename(columns={"Sample": "donor"}),
    on="donor",
    how="left"
)

c:\Users\ankit\miniconda3\envs\scanpy\Lib\functools.py:982: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [7]:
# View the updated obs columns and head
print(adata_merged.obs.columns.tolist())
print(adata_merged.obs.head())

['sample_title', 'donor', 'assay', 'prep', 'gsm_id', 'condition', 'dataset', 'batch', 'n_counts', 'n_genes', 'Sex', 'Cause_of_death']
          sample_title    donor      assay prep      gsm_id condition  \
0  Single cell Donor 1  Donor_1  scRNA-seq   SC  GSM3178784   Healthy   
1  Single cell Donor 1  Donor_1  scRNA-seq   SC  GSM3178784   Healthy   
2  Single cell Donor 1  Donor_1  scRNA-seq   SC  GSM3178784   Healthy   
3  Single cell Donor 1  Donor_1  scRNA-seq   SC  GSM3178784   Healthy   
4  Single cell Donor 1  Donor_1  scRNA-seq   SC  GSM3178784   Healthy   

     dataset       batch  n_counts  n_genes Sex Cause_of_death  
0  GSE185477  GSM3178784         1        1   F       Aneurysm  
1  GSE185477  GSM3178784         2        1   F       Aneurysm  
2  GSE185477  GSM3178784         1        1   F       Aneurysm  
3  GSE185477  GSM3178784       137       56   F       Aneurysm  
4  GSE185477  GSM3178784       216       50   F       Aneurysm  


In [23]:
# View the counts of scRNA-seq vs snRNA-seq for Donor 1
print(adata_merged[adata_merged.obs["donor"] == "Donor_1"].obs["assay"].value_counts())

assay
snRNA-seq    2609052
scRNA-seq     303271
Name: count, dtype: int64
